<a href="https://colab.research.google.com/github/lerkalarionova2018-rgb/nlp-homeworks/blob/main/%D0%9B%D0%B0%D1%80%D0%B8%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0_%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install huggingface_hub -q
!pip install transformers datasets evaluate accelerate -q

import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

# Проверка GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [2]:
# Загрузка датасета AG News
dataset = load_dataset("fancyzhx/ag_news")
print(f"Датасет загружен.")
print(f"Train: {len(dataset['train'])} примеров")
print(f"Test: {len(dataset['test'])} примеров")
print(f"\nСтруктура датасета:")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен.
Train: 120000 примеров
Test: 7600 примеров

Структура датасета:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [3]:
# Просмотр примера из датасета
print("Пример из обучающей выборки:")
print(dataset['train'][0])
print(f"\nКлассы: 0 - World, 1 - Sports, 2 - Business, 3 - Sci/Tech")

Пример из обучающей выборки:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Классы: 0 - World, 1 - Sports, 2 - Business, 3 - Sci/Tech


In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

print(f"Модель {model_name} загружена на {device}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Модель distilbert-base-uncased загружена на cuda


In [9]:
# Токенизация с уменьшением выборки
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

# Токенизация полного датасета
print("Токенизация данных...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Берем только часть данных для обучения
TRAIN_SAMPLES = 3000
TEST_SAMPLES = 1000

# Создаем уменьшенные датасеты
train_dataset_small = tokenized_datasets["train"].shuffle(seed=42).select(range(min(TRAIN_SAMPLES, len(tokenized_datasets["train"]))))
eval_dataset_small = tokenized_datasets["test"].shuffle(seed=42).select(range(min(TEST_SAMPLES, len(tokenized_datasets["test"]))))

print(f"Оригинальный train: {len(tokenized_datasets['train'])} → Уменьшенный: {len(train_dataset_small)}")
print(f"Оригинальный test: {len(tokenized_datasets['test'])} → Уменьшенный: {len(eval_dataset_small)}")

Токенизация данных...


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Оригинальный train: 120000 → Уменьшенный: 3000
Оригинальный test: 7600 → Уменьшенный: 1000


In [10]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [13]:
training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    logging_steps=50,
    fp16=True,
)

print("Параметры обучения настроены:")
print(f"- Learning rate: {training_args.learning_rate}")
print(f"- Batch size: {training_args.per_device_train_batch_size}")
print(f"- Epochs: {training_args.num_train_epochs}")
print(f"- FP16: {training_args.fp16}")
print(f"- Train samples: {len(train_dataset_small)}")
print(f"- Eval samples: {len(eval_dataset_small)}")

Параметры обучения настроены:
- Learning rate: 2e-05
- Batch size: 16
- Epochs: 3
- FP16: True
- Train samples: 3000
- Eval samples: 1000


In [15]:
import time
from transformers import Trainer, DataCollatorWithPadding

start_time = time.time()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_small,
    eval_dataset=eval_dataset_small,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Начинаем обучение модели на уменьшенной выборке...")

trainer.train()


end_time = time.time()
training_time = end_time - start_time

print(f" Обучение завершено!")
print(f"Время обучения: {training_time:.2f} секунд ")

Начинаем обучение модели на уменьшенной выборке...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.089913,0.367799,0.901000
2,0.063734,0.372443,0.907000
3,0.037763,0.394001,0.907000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


 Обучение завершено!
Время обучения: 68.30 секунд 


In [16]:
print("\n Оценка модели на тестовой выборке...")
eval_results = trainer.evaluate()

print("РЕЗУЛЬТАТЫ ОЦЕНКИ:")
print(f"Loss на тесте: {eval_results['eval_loss']:.4f}")
print(f"Accuracy на тесте: {eval_results['eval_accuracy']:.4f}")
print(f"Точность модели: {eval_results['eval_accuracy']*100:.2f}%")
print("=" * 50)

print("\n Проверка стабильности на дополнительных 100 примерах...")
random_100 = tokenized_datasets["test"].shuffle(seed=123).select(range(100))
stability_results = trainer.evaluate(random_100)
print(f"Accuracy на 100 случайных примерах: {stability_results['eval_accuracy']*100:.2f}%")


 Оценка модели на тестовой выборке...


РЕЗУЛЬТАТЫ ОЦЕНКИ:
Loss на тесте: 0.3724
Accuracy на тесте: 0.9070
Точность модели: 90.70%

 Проверка стабильности на дополнительных 100 примерах...
Accuracy на 100 случайных примерах: 91.00%


In [17]:
model_save_path = "./ag_news_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"\n Модель сохранена в папку: {model_save_path}")
print(f"Сохраненные файлы:")
import os
for file in os.listdir(model_save_path):
    if not file.startswith('.'):
        size = os.path.getsize(os.path.join(model_save_path, file))
        print(f"  - {file} ({size:,} bytes)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Модель сохранена в папку: ./ag_news_model
Сохраненные файлы:
  - checkpoint-188 (4,096 bytes)
  - tokenizer_config.json (322 bytes)
  - config.json (855 bytes)
  - tokenizer.json (711,494 bytes)
  - model.safetensors (267,838,720 bytes)
  - checkpoint-564 (4,096 bytes)
  - training_args.bin (5,201 bytes)
  - checkpoint-376 (4,096 bytes)


In [18]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model_save_path,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Словарь для преобразования меток в названия категорий
label_map = {
    "LABEL_0": " World (Мир)",
    "LABEL_1": " Sports (Спорт)",
    "LABEL_2": " Business (Бизнес)",
    "LABEL_3": " Sci/Tech (Наука и технологии)"
}

# Тестовые новости
test_news = [
    # Sci/Tech
    "Apple announces new iPhone with advanced AI capabilities and longer battery life",
    # Sports
    "Manchester United wins Premier League title after dramatic final match",
    # World
    "Global climate summit reaches historic agreement on carbon emissions reduction",
    # Business
    "Stock markets rally as Federal Reserve signals interest rate cuts",
    # Дополнительные тесты
    "Scientists discover new exoplanet in habitable zone of distant star system",
    "Olympic champion announces retirement after winning gold medal"
]

print(" ТЕСТИРОВАНИЕ МОДЕЛИ НА ТЕСТОВЫХ НОВОСТЯХ")

for i, news in enumerate(test_news, 1):
    result = classifier(news)[0]
    predicted_label = result['label']
    confidence = result['score']
    category = label_map.get(predicted_label, predicted_label)

    print(f"\n Новость {i}:")
    print(f"   Текст: {news[:80]}..." if len(news) > 80 else f"   Текст: {news}")
    print(f"   Предсказанный класс: {category}")
    print(f"   Уверенность: {confidence:.4f} ({confidence*100:.2f}%)")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

 ТЕСТИРОВАНИЕ МОДЕЛИ НА ТЕСТОВЫХ НОВОСТЯХ

 Новость 1:
   Текст: Apple announces new iPhone with advanced AI capabilities and longer battery life
   Предсказанный класс:  Sci/Tech (Наука и технологии)
   Уверенность: 0.9977 (99.77%)

 Новость 2:
   Текст: Manchester United wins Premier League title after dramatic final match
   Предсказанный класс:  Sports (Спорт)
   Уверенность: 0.9985 (99.85%)

 Новость 3:
   Текст: Global climate summit reaches historic agreement on carbon emissions reduction
   Предсказанный класс:  Sci/Tech (Наука и технологии)
   Уверенность: 0.9923 (99.23%)

 Новость 4:
   Текст: Stock markets rally as Federal Reserve signals interest rate cuts
   Предсказанный класс:  Business (Бизнес)
   Уверенность: 0.9932 (99.32%)

 Новость 5:
   Текст: Scientists discover new exoplanet in habitable zone of distant star system
   Предсказанный класс:  Sci/Tech (Наука и технологии)
   Уверенность: 0.9969 (99.69%)

 Новость 6:
   Текст: Olympic champion announces retirement af